* Nhớ bật Internet và GPU

- B1: Chạy cell 1 - Các thư viện trước.

- B2: Bấm nút ba chấm, chọn restart & clear shells outputs để reset thư viện.

- B3: Chạy lại cell 1 lần nữa.

- B4: Chạy cell 2, nó ra cái link là xong.

In [2]:
# 1. Gỡ cài đặt các bản lỗi và thư viện dư thừa (Thêm cupy vào đây)
!pip uninstall -y torch torchvision torchaudio xformers cupy cupy-cuda12x cupy-cuda11x

# 2. Cài đặt các thư viện lõi AI (chuẩn CUDA 12.1)
!pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q xformers==0.0.28.post3 --index-url https://download.pytorch.org/whl/cu121

# 3. Cài đặt các thư viện AI vệ tinh
!pip install -q diffusers transformers accelerate peft pyngrok

# 4. Cài đặt bộ 3 thư viện hay gây xung đột (Đã cập nhật rembg lên 2.0.60)
!pip install -q "numpy==1.26.4" "rembg[gpu]==2.0.60" "pillow==9.5.0"

!pip install -q diffusers transformers accelerate peft pyngrok deep-translator

Found existing installation: torch 2.5.1+cu121
Uninstalling torch-2.5.1+cu121:
  Successfully uninstalled torch-2.5.1+cu121
Found existing installation: torchvision 0.20.1+cu121
Uninstalling torchvision-0.20.1+cu121:
  Successfully uninstalled torchvision-0.20.1+cu121
Found existing installation: torchaudio 2.5.1+cu121
Uninstalling torchaudio-2.5.1+cu121:
  Successfully uninstalled torchaudio-2.5.1+cu121
Found existing installation: xformers 0.0.28.post3
Uninstalling xformers-0.0.28.post3:
  Successfully uninstalled xformers-0.0.28.post3
^C
ERROR: Operation cancelled by user


In [3]:
import os
from pyngrok import ngrok

print("🧹 Cleaning up port 5050...")
ngrok.kill()
os.system("fuser -k 5050/tcp")

# ==========================================
# 🚀 SERVER KAGGLE (DUAL GPU + QUEUE OPTIMIZED)
# ==========================================
import torch, io
from queue import Queue
from threading import Thread
from flask import Flask, request, send_file
from deep_translator import GoogleTranslator
from diffusers import StableDiffusionXLPipeline, EulerDiscreteScheduler
from PIL import Image
from rembg import remove, new_session
import time

# --- CONFIG ---
NGROK_TOKEN = "35QROiAHAwwrO5DrCxdKFqPPMP4_3U66DPiMUe46aRawZ5pjT"
MODEL_ID = "Lykon/dreamshaper-xl-lightning" # Model hỗ trợ tốt IP-Adapter và vẽ phong cảnh

# --- 1. TẢI CÔNG CỤ TÁCH NỀN (CHẠY TRÊN CPU ĐỂ NHƯỜNG VRAM CHO ẢNH) ---
print("🚀 Loading rembg (CPU mode)...")
rembg_session = new_session("u2netp", providers=['CPUExecutionProvider'])

# --- 2. NẠP MODEL VÀO CẢ 2 GPU (SONG SONG) ---
print("🚀 Loading SDXL on DUAL GPUs...")
gpu_queue = Queue()

for gpu_id in [0, 1]:
    device = f"cuda:{gpu_id}"
    print(f" -> Đang nạp model vào {device}...")
    
    pipe = StableDiffusionXLPipeline.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        variant="fp16"
    )
    
    pipe.scheduler = EulerDiscreteScheduler.from_config(
        pipe.scheduler.config,
        timestep_spacing="trailing",
        original_inference_steps=4
    )
    
    pipe.safety_checker = None
    pipe.requires_safety_checker = False
    
    # Nạp IP-Adapter (nếu có sử dụng ảnh mồi)
    try:
        pipe.load_ip_adapter("h94/IP-Adapter", subfolder="sdxl_models", weight_name="ip-adapter_sdxl.bin")
    except Exception as e:
        print(f" [!] Bỏ qua IP-Adapter trên {device}: {e}")
        
    pipe.to(device)
    
    # Xếp GPU đã sẵn sàng vào hàng đợi
    gpu_queue.put(pipe)

print("✅ Đã khởi tạo xong 2 GPU!")

# --- 3. SERVER FLASK ---
app = Flask(__name__)

@app.route("/api/image", methods=["POST"])
def gen_img():
    start_server = time.time()
    
    # 1. Nhận thông số từ Backend
    raw_prompt = request.form.get("prompt", "")
    image_type = request.form.get("image_type", "background").lower() 
    quality = request.form.get("quality", "medium").lower() 

    # 2. Dịch prompt sang tiếng Anh
    try:
        prompt_en = GoogleTranslator(source='vi', target='en').translate(raw_prompt)
    except Exception:
        prompt_en = raw_prompt 
    
    # 3. Ép xung Kích thước và Số bước vẽ
    size_map = {
        "background": {"low": (512, 384), "medium": (896, 512), "high": (1280, 720)},
        "npc": {"low": (384, 512), "medium": (512, 896), "high": (768, 1280)},
        "item": {"low": (384, 384), "medium": (512, 512), "high": (1024, 1024)}
    }
    steps_map = {"low": 2, "medium": 3, "high": 5}

    w, h = size_map.get(image_type, size_map["background"]).get(quality, size_map["background"]["medium"])
    steps = steps_map.get(quality, 3)

    # 4. Xử lý ảnh mồi IP-Adapter
    ip_image = None
    style_scale = 0.0
    if 'image' in request.files and request.files['image'].filename != '':
        try:
            ip_image = Image.open(request.files['image']).convert("RGB")
            style_scale = float(request.form.get("style_scale", 0.6))
        except: 
            pass

    # ==========================================
    # 🚀 LUỒNG XỬ LÝ ĐA GPU (QUEUE)
    # ==========================================
    # Xin 1 GPU đang rảnh từ hàng đợi (Nếu cả 2 đều bận, luồng sẽ đứng chờ ở đây)
    pipe = gpu_queue.get()
    
    try:
        with torch.inference_mode():
            pipe_kwargs = {
                "prompt": f"{prompt_en}, masterpiece, best quality, highly detailed", 
                "num_inference_steps": steps, 
                "guidance_scale": 0.0, 
                "width": w,
                "height": h
            }

            if ip_image is not None:
                pipe.set_ip_adapter_scale(style_scale)
                pipe_kwargs["ip_adapter_image"] = ip_image
            else:
                dummy_img = Image.new("RGB", (224, 224), (0, 0, 0))
                pipe.set_ip_adapter_scale(0.0)
                pipe_kwargs["ip_adapter_image"] = dummy_img

            start_sdxl = time.time()
            # Tiến hành vẽ ảnh bằng GPU hiện tại
            image = pipe(**pipe_kwargs).images[0]
            time_sdxl = time.time() - start_sdxl
            
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        return {"error": "OOM"}, 503
    finally:
        # ⚠️ CỰC KỲ QUAN TRỌNG: Trả GPU lại cho hàng đợi sau khi vẽ xong
        gpu_queue.put(pipe)
        torch.cuda.empty_cache()
        
    # ==========================================

    # 5. Xử lý Tách nền (Rembg)
    start_rembg = time.time() 
    if image_type in ["npc", "item"]:
        image = remove(image, session=rembg_session)
    time_rembg = time.time() - start_rembg
    
    # 6. Mã hóa và gửi về Python
    start_encode = time.time()
    buf = io.BytesIO()
    image.save(buf, format="PNG", optimize=True)
    buf.seek(0)
    time_encode = time.time() - start_encode

    total_server_time = time.time() - start_server
    print(f"⏱️ [{pipe.device} - {image_type.upper()}] TỔNG THỜI GIAN: {total_server_time:.2f} s (Vẽ: {time_sdxl:.2f}s | Nền: {time_rembg:.2f}s)")
    
    return send_file(buf, mimetype='image/png')


# --- 4. START SERVER ---
def start():
    ngrok.set_auth_token(NGROK_TOKEN)
    domain="unspelt-nonbrutally-eleanore.ngrok-free.dev"
    public_url = ngrok.connect(5050, domain="unspelt-nonbrutally-eleanore.ngrok-free.dev").public_url

    print("\n=======================================================")
    print(f"🌍 BACKEND URL CỐ ĐỊNH: {public_url}")
    print("=======================================================\n")

    app.run(host="0.0.0.0", port=5050, use_reloader=False)

Thread(target=start).start()

🧹 Cleaning up port 5050...


KeyboardInterrupt: 